# SE2_data_process
This notebook will be used to generate bathymetric surfaces from previously downloaded hydrographic survey point cloud data. Four interpolation methods are implemented:
1. Triangulated Irregular Network (TIN)
2. Natural Neighbor
3. Ordinary kriging (OK)
4. Regrssion Kriging (RK)

Additional preprocessing steps includes a mean data decimation to reduce the point clouds to 15,000 points to emulate a common approach in bathymetric point cloud processing, and to maintain Kriging numerical stability.

# Overview:
## Input data description:
Data is stored in .gdb file format, which is read using *fiona* and *geopandas*. Because eHydro covers many regions of the US, data is stored in varying forms of the State Plane Coordinate system for the region the data was aquired. Within the .gdb, three layers may be used: *SurveyPoint* for a pre-processed point clouds, *SurveyPoint_HD* for a pre-processed dense point clouds, and *SurveyJob* for a boundary shapefile for the survey. Within the two available point cloud layers, the xLocation, yLocation, and Depth (Z_use) are reported relative to a local Mean Lowest Low Water (MLLW) datum.

## Outputs:
Each point cloud will be processed into Cloud Optimized GeoTIFF (COG) format. The COGs will be saved in their original CRS relative to the corresponding .gdb file which contained the point cloud. The spatial resolution for each COG is 10 feet, to correspond with the inputs used in the US Army Corps of Engineers Corps Shoaling Analysis Tool (CSAT). Additionally, with each COG, a validation CSV is provided which stores the sampled depth from the point cloud, the predicted depth, and residual at each point within the raw point cloud. This CSV will be used to analyze bias and compare interpolation performance.

## Modelling framework:
Processed data will be compatible with most software, as the two data outputs are stored in open COG and CSV formats.

## Data processing steps:
1. Data decimation to a target number of points using *Verde.BlockReduce*. A reduction method may be specified, but Median is default
2. Specify and fit multiple variogram shape functions, using Cressie or Matheron weighted least squares approach to solve for nugget, range, and sill. The Aikaike Information Criterion is used to select the best fit empirical variogram for the kriging model
3. TIN, Natural Neighbor, Ordinary Kriging, and Regression Kriging employed on decimated point cloud to generate COGs
4. Raw dense point clouds sampled along predicted rasters for validation

## How to use this notebook:
This notebook will read data from ```raw_data``` folder, or a specified data storage folder. A user may specify the output grid resolution, the specific variogram models to test, which least square solution to use for empirical variogram fitting, what reduction method to use for data decimation, and whether or not to plot the COGs as they are generated.



In [ ]:
%reload_ext autoreload
%autoreload 2
import os
import sys
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd
import geopandas as gpd
from pathlib import Path
import gstools as gs
import fiona
import matplotlib.pyplot as plt

from interpolators.TIN import TIN
from interpolators.NaturalNeighbor import NaturalNeighbor
from interpolators.KNeighbors_IDW import IDW
from interpolators.RBF import RBF
from interpolators.isoAutoKrige import isoAutoKrige

import processing_help
import numpy as np
from scipy.spatial import cKDTree, ConvexHull
from sklearn.preprocessing import StandardScaler

import warnings
# remove future warning of spline mindist since we aren't even providing it anyways
warnings.filterwarnings('ignore', message='.*mindist parameter.*', category=FutureWarning)     

In [ ]:
DATA_DIR = Path(os.getcwd()).parent / 'raw_data'
# DATA_DIR = Path("/mnt/teamgroup/AutoDBM/paper_work/raw_data")

survey_gdbs = list(DATA_DIR.glob('*/*/*.gdb'))

# First, we need to calculate survey characteristics so we can understand how they interact with the accuracy (RMSE) of each interpolation method. These include:
- log-transformed sounding density for each survey
- log-transformed standard deviation of the sounding measurements (depth)
- log transformed anisotropy in the distribution of the soundings
- log tranformed 3x3 window Terrain Roughness Index

In [ ]:
# ALL FEATURES NEED TO BE LOG TRANSFORMED BEFORE STANDARD SCALING AND THEN WE CAN USE THE MAXMIN SELECTOR
# NOTE: GAM and plot for the selected surveys should be in the original scale, and not log-transformed or scaled

def calculate_survey_characteristics(x, y, z, radius=3.0):
    """
    Function to calculate the standard deviation of depth, log-transformed density of the point cloud, 
    (global) anisotropy of the point cloud (e.g., anisotropy in the shape of the survey),
    and a mean Terrain Roughness Index based on a fixed radius search within the point cloud.
    """
    coords = np.column_stack([x, y])
    
    # Log-density
    hull = ConvexHull(coords)
    density = len(x) / hull.volume
    # log_density = np.log10(density)
    
    # Depth std
    depth_std = np.std(z, ddof=1)
    # log_depth_std = np.log1p(depth_std)
    
    # Anisotropy
    cov = np.cov(x, y)
    eigs = np.sort(np.linalg.eigvalsh(cov))[::-1]
    anisotropy = (eigs[0] - eigs[1]) / (eigs[0] + eigs[1])
    # log_anisotropy = np.log1p(anisotropy)
    
    # TRI
    tree = cKDTree(coords)
    tri_values = []
    for i in range(len(x)):
        h00 = z[i]  # Central point elevation
        neighbors = tree.query_ball_point(coords[i], r=radius)
        
        if len(neighbors) > 1:  # Ensure there are enough neighbors to calculate a meaningful TRI
            local_elevations = z[neighbors]
            deviations = np.abs(local_elevations - h00)
            tri = np.mean(deviations)
            tri_values.append(tri)
        else:
            # If there are not enough neighbors, you can choose to append a default value or skip
            tri_values.append(0.0)
    
    roughness = np.mean(tri_values)
    # log_roughness = np.log1p(roughness)
    
    return density, depth_std, anisotropy, roughness

In [ ]:
survey_characteristics = {}

for file in survey_gdbs:
    if 'SurveyPoint_HD' in fiona.listlayers(file):
        data = gpd.read_file(file, layer = 'SurveyPoint_HD')
    else:
        data = gpd.read_file(file, layer = 'SurveyPoint')
    
    if len(data) < 10000:
        # print(f"Point cloud for {file.stem} is less than 10,000 points.")
        continue
    else:
        if 'xLocation' in data.columns and 'yLocation' in data.columns and 'Z_use' in data.columns:
            data_x = data.xLocation.values
            data_y = data.yLocation.values
            data_z = data.Z_use.values
        else:
            continue

        # searches with fixed radius of 3.0 if 'radius' not specified
        survey_characteristics[file.stem] = calculate_survey_characteristics(data_x, data_y, data_z)
        print(f"{file.stem} processing completed...")

In [ ]:
cols = ["density", "depth_std", "anisotropy", "roughness"]

survey_characteristics_df = (
    pd.DataFrame.from_dict(
        survey_characteristics,
        orient="index",
        columns=cols,
    )
    .reset_index()
    .rename(columns={"index": "survey"})
)

survey_characteristics_df["survey"] = survey_characteristics_df["survey"].astype(str)
survey_characteristics_df.head()

survey_characteristics_df.to_csv(DATA_DIR / "survey_characteristics_df.csv")

In [ ]:
pd.read_csv(DATA_DIR / "survey_characteristics_df.csv")

In [ ]:
# TODO: log-transform these variables as they are heavily skewed


# make a standardized version of both dataframes for easier visualization of which surveys we have sampled
std_scaler = StandardScaler()
df_scaled = std_scaler.fit_transform(survey_characteristics_df[cols].to_numpy())

survey_characteristics_scaled = pd.DataFrame(df_scaled, columns = cols)

row_mask = (survey_characteristics_scaled[cols].abs() < 3).all(axis=1)

survey_characteristics_scaled = survey_characteristics_scaled.loc[row_mask].reset_index(drop=True)

{
    "dropped_rows": int((~row_mask).sum()),
    "remaining_rows": len(survey_characteristics_scaled),
    "shape": survey_characteristics_scaled.shape,
}

In [ ]:
scaled_features = survey_characteristics_scaled[["log_density", "depth_std", "anisotropy", "roughness"]].values
features = survey_characteristics_df[["log_density", "depth_std", "anisotropy", "roughness"]].values

# Greedy MaxMin selection
def select_diverse(X, n=100):
    n_samples = len(X)
    selected = [np.random.randint(n_samples)]  # random seed point
    min_dists = np.full(n_samples, np.inf)

    for _ in range(n - 1):
        last = X[selected[-1]]
        dists = np.linalg.norm(X - last, axis=1)
        min_dists = np.minimum(min_dists, dists)
        min_dists[selected] = -1  # exclude already chosen
        selected.append(np.argmax(min_dists))

    return selected

scaled_indices = select_diverse(scaled_features, n=500)
indices = select_diverse(features, n=500)
diverse_subset_scaled = survey_characteristics_scaled.iloc[scaled_indices]
diverse_subset = survey_characteristics_df.iloc[indices]

In [ ]:
features = ["log_density", "depth_std", "anisotropy", "roughness"]
n_features = len(features)

fig, axes = plt.subplots(n_features, n_features, figsize=(12, 12))

for i in range(n_features):
    for j in range(n_features):
        ax = axes[i, j]
        if i == j:
            # Marginal distributions on the diagonal
            ax.hist(survey_characteristics_scaled[features[i]], bins=30, color="lightgray", density=True)
            ax.hist(diverse_subset_scaled[features[i]], bins=20, color="firebrick", alpha=0.7, density=True)
        elif i > j:
            # Scatter in lower triangle
            ax.scatter(survey_characteristics_scaled[features[j]], survey_characteristics_scaled[features[i]],
                       c="lightgray", s=2, alpha=0.3)
            ax.scatter(diverse_subset_scaled[features[j]], diverse_subset_scaled[features[i]],
                       c="firebrick", s=8, alpha=0.6)
        else:
            ax.axis("off")  # Upper triangle empty to reduce clutter

        if j == 0 and i != 0:
            ax.set_ylabel(features[i])
        if i == n_features - 1:
            ax.set_xlabel(features[j])

plt.tight_layout()

In [ ]:
os.makedirs(DATA_DIR.parent / 'analysis_data', exist_ok=True)
diverse_subset.to_csv(DATA_DIR.parent / 'analysis_data' / 'survey_characteristics.csv', index=False)

# Now, we can generate surfaces for the most diverse surveys from our dataset
- For publication, we are aiming for 100+ surveys from across the US

In [ ]:
tgt_survey_gdbs = []

for survey in diverse_subset.survey.values:
    tgt_survey_gdbs.append(list(DATA_DIR.glob(f'*/*{survey}.gdb')))

survey_gdbs = np.unique(tgt_survey_gdbs)

In [ ]:
vario_models = {
    "Gaussian": gs.Gaussian,
    "Exponential": gs.Exponential,
    "Matern": gs.Matern,
    "Stable": gs.Stable,
    "Rational": gs.Rational,
    "Circular": gs.Circular,
    "Spherical": gs.Spherical,
    "SuperSpherical": gs.SuperSpherical,
    "JBessel": gs.JBessel,
}

# least square estimation scheme for automatic variogram fitting
# 'cressie' or 'matheron'. Cressie tends to be more robust to outliers
estimator = 'cressie'

# all data from eHydro is projected in state plance coordinates
# measurements units referenced to US Survey Feet
grid_res = 10.0 # ft

# for example on a not so big computer
# NOTE: Kriging is very computationally heavy, as is global spline fitting. A PC with high availabel RAM is preferred
# even with ~ 90G of RAM, I can only make it to 15k points sometimes
num_pts= 10000

reduction_method = 'median'     # median to remove outliers, min to get shoalest data for more accurately representing potential shoals. Will test and compare results using median, mean, minimum and nav_safe to know which is best

plots = True               # if the outputs of the interpolation methods will be showcased in this notebook using Matplotlib

# Generate surfaces for a single target survey

In [ ]:
for survey in survey_gdbs:
    if 'SurveyPoint_HD' in fiona.listlayers(survey):
        survey_pts = gpd.read_file(survey, layer = 'SurveyPoint_HD')
    else:
        survey_pts = gpd.read_file(survey, layer = 'SurveyPoint')
    
    # commented out for runs without validation
    # x = survey_pts.xLocation.values
    # y = survey_pts.yLocation.values
    # z = survey_pts.Z_use.values
    
    x, y, z, x_raw, y_raw, z_raw = processing_help.decimate_survey_points(survey_pts, tgt_num_pts=num_pts, reduction_method=reduction_method)
        
    TINinterpolator = TIN(
            tgt_survey = survey,
            reduction_method = reduction_method,
            x=x, y=y, z=z,
            grid_res=grid_res,
            tgt_num_pts=num_pts,
            plot_outputs=plots,
            n_jobs=4
        )
    
    tin_path = TINinterpolator.generate()
    _,_ = processing_help.validate_bathymetric_surface(tin_path, x_raw, y_raw, z_raw)
    
    NatNinterpolator = NaturalNeighbor(
            tgt_survey = survey, 
            reduction_method = reduction_method,
            x=x, y=y, z=z, 
            grid_res=grid_res,
            tgt_num_pts=num_pts,
            plot_outputs=plots,
            n_jobs=4
        )
        
    natn_path = NatNinterpolator.generate()
    _,_ = processing_help.validate_bathymetric_surface(natn_path, x_raw, y_raw, z_raw)
    
    IDWinterpolator = IDW(
            tgt_survey=survey,
            reduction_method=reduction_method,
            x=x, y=y, z=z,
            grid_res=grid_res,
            tgt_num_pts=num_pts,
            plot_outputs=plots,
            n_jobs=4
        )
        
    idw_path = IDWinterpolator.generate()
    _,_ = processing_help.validate_bathymetric_surface(idw_path, x_raw, y_raw, z_raw)

    # NOTE: this is commented out since the grid search for the RBF methods can be computationally heavy
    # RBFinterpolator = RBF(
    #         tgt_survey=survey,
    #         reduction_method=reduction_method,
    #         x=x, y=y, z=z, 
    #         grid_res=grid_res,
    #         tgt_num_pts=num_pts,
    #         plot_outputs=plots,
    #         n_jobs=4
    #     )
    
    # rbf_path = RBFinterpolator.generate(auto_tune=True)
    # _,_ = processing_help.validate_bathymetric_surface(rbf_path, x_raw, y_raw, z_raw)
    
    for detrend_bool in [False, True]:
        for chunk in [False, True]:
            
            isoKrige_AIC = isoAutoKrige(
                            tgt_survey = survey, 
                            vario_models=vario_models,
                            reduction_method=reduction_method,
                            x=x, y=y, z=z, 
                            spline_damping=1e-1,                                            
                            vario_estimator = estimator,            
                            chunked_kriging = chunk,
                            grid_res=grid_res,
                            detrend=detrend_bool,
                            tgt_num_pts=num_pts,
                            plot_outputs=plots,
                            n_jobs=4
                    )
            krige_path_AIC, _ = isoKrige_AIC.generate()
            _,_ = processing_help.validate_bathymetric_surface(krige_path_AIC, x_raw, y_raw, z_raw)

# Prepare the validation and survey characteristics for use in R for interpolation accuracy comparison

In [ ]:
validation_results = list(DATA_DIR.glob('*/*.csv'))

rmse_agg = {}

# Build a set of known survey names for matching
known_surveys = set(diverse_subset['survey'].values)

for file in validation_results:

    if reduction_method not in str(file):
        print(file)
        continue
    else:
        val_file = pd.read_csv(file)
        folder = file.parts[-2]
        
        # Find the matching survey name from diverse_subset
        survey = None
        for s in known_surveys:
            if folder.endswith(s):
                survey = s
                break
        if survey is None:
            survey = folder  # fallback: keep full folder name
        
        rmse_agg[f'{survey}/{file.stem.replace("_validation", "")}.tif'] = np.sqrt(np.nanmean((val_file.True_Depth.to_numpy() - val_file.Predicted_Depth.to_numpy())**2))


rmse_agg_df = (
    pd.DataFrame.from_dict(
        rmse_agg,
        orient="index",
        columns=["mean_rmse"],
    )
    .reset_index()
    .rename(columns={"index": "survey_surface"})
)

rmse_agg_df

In [ ]:
# Split survey_surface into survey name and method
rmse_agg_df[['survey', 'surface']] = rmse_agg_df['survey_surface'].str.split('/', expand=True)
rmse_agg_df['surface'] = rmse_agg_df['surface'].str.replace('_depth.tif', '', regex=False)

# Pivot to wide format: 1 row per survey, 1 column per method
rmse_wide = rmse_agg_df.pivot(index='survey', columns='surface', values='mean_rmse').reset_index()
rmse_wide.columns.name = None  # clean up the column axis name

In [ ]:
common = set(diverse_subset['survey']) & set(rmse_wide['survey'])
rmse_wide_filtered = rmse_wide[rmse_wide["survey"].isin(common)].copy()
rmse_wide_filtered.to_csv(DATA_DIR.parent / 'analysis_data' / 'aggregated_rmse.csv', index=False)